# Summary

Search the Bedrock Knowledge base

In [2]:
import os, sys
import pandas as pd
import re
import json
import time

# AWS Python
import boto3

utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Construct a AWS Bedrock Retrieval and Generate class object

In [30]:
class BedrockKBRetriever:
    """
    Class to demonstrate AWS Bedrock Knowledge Base retrieve and retrieve and generate results.
    """

    def __init__(self,
                 aws_profile: str,
                 aws_region: str,
                 kb_id: str):

        self.aws_profile = aws_profile
        self.aws_region = aws_region
        self.kb_id = kb_id

        # Retrieval configuration
        self.num_srch_res = 5
        self.s3_bucket = "rag-search-tests"
        self.s3_path = "documents/"
        self.s3_doc_loc = "s3://{}/{}".format(self.s3_bucket,
                                              self.s3_path)

        # Retrieval and Generation configuration
        self.aws_account_id = os.environ["AWS_ACCOUNT_ID"]
        # self.inference_model = "amazon.nova-lite-v1:0" #?
        # self.inference_model = "anthropic.claude-sonnet-4-6" # Requires Anthropic request form
        # self.inference_model = "google.gemma-3-4b-it" # Not available
        # self.inference_model = "meta.llama4-scout-17b-instruct-v1:0"
        self.inference_model = "anthropic.claude-sonnet-4-5-20250929-v1:0"
        # self.inference_model = "amazon.nova-pro-v1:0"
        # self.inference_model = "google.gemma-3-12b-it"
        self.model_arn = "arn:aws:bedrock:{}:{}:inference-profile/us.{}".format(self.aws_region,
                                                                                self.aws_account_id,
                                                                                self.inference_model)
        self.search_sleep_time = 1.5 # pause time between queries
        self.ai_sleep_time = 1.5 # pause time between queries

        # Outputs
        self.df_ret_res = pd.DataFrame()
        self.df_query_finds = pd.DataFrame()
        self.df_rag_res = pd.DataFrame()
        self.bedrock_models = pd.DataFrame() # AWS foundation models - requires get_bedrock_foundational_models method
        self.rag_responses = []

        # Establish an AWS client
        self.set_aws_client()

    def set_aws_client(self):
        """
        Create a AWS client
        :return:
        """

        # Initialize Bedrock Agent Runtime client for querying
        session = boto3.Session(profile_name=self.aws_profile)

        # Bedrock client
        self.br_rt = session.client('bedrock-agent-runtime',
                                    region_name=self.aws_region)

        # S3 client
        self.s3 = session.client('s3',
                                 region_name=self.aws_region)

    def get_bedrock_foundational_models(self):
        """
        Get a list of foundation models available through Bedrock
        :return:
        """

        # Initialize a bedrock client
        session = boto3.Session(profile_name=self.aws_profile)
        self.br = session.client('bedrock', region_name=self.aws_region)

        response = self.br.list_foundation_models()
        self.response_models = response

        models_rows = []
        for model in response['modelSummaries']:
            models_rows.append(dict(model_name=model['modelName'],
                                    model_id=model['modelId'],
                                    model_provider=model['providerName'],
                                    input_modalities=model['inputModalities'],
                                    output_modalities=model['outputModalities'],
                                    infer_types=model['inferenceTypesSupported'],
                                    model_lifecycle=model['modelLifecycle']
                                    )
                               )

        self.bedrock_models = pd.DataFrame(data=models_rows)
        self.bedrock_models = self.bedrock_models.sort_values(by="model_provider")
        self.bedrock_models = self.bedrock_models.reset_index(drop=True)

    def retrieve_query_results(self, queries: list, metadata_filters: dict = None):
        """
        General search for AWS Bedrock with dynamic metadata filtering.

        :param queries: List of search strings.
        :param metadata_filters: Dict of key-value pairs, e.g.,
                                 {'source_type': ['pdf', 'txt'], 'document_index': [1, 2]}
        """

        # 1. Build the dynamic filter expression
        filter_expression = None

        if metadata_filters:
            filter_list = []
            for key, values in metadata_filters.items():
                if values:  # Only add if there are values to filter by
                    filter_list.append({
                        'in': {
                            'key': key,
                            'value': values
                        }
                    })

            # If multiple filters exist, wrap them in an 'and' operator
            if len(filter_list) > 1:
                filter_expression = {'and': filter_list}
            elif len(filter_list) == 1:
                filter_expression = filter_list[0]

        # 2. Set retrieval configurations
        ret_config = {
            'vectorSearchConfiguration': {
                'numberOfResults': self.num_srch_res
            }
        }

        if filter_expression:
            ret_config['vectorSearchConfiguration']['filter'] = filter_expression

        search_results = []
        for query in queries:

            # Set retrieval configurations
            ret_query_param = {'text': query
                               }

            response = self.br_rt.retrieve(knowledgeBaseId=self.kb_id,
                                           retrievalQuery=ret_query_param,
                                           retrievalConfiguration=ret_config
                                           )

            for result in response["retrievalResults"]:

                # Get a list of passages
                regex_pattern = r"\[(\d+)\]"

                passages = re.findall(regex_pattern, result['content']['text'], flags=re.DOTALL)

                # Clean up whitespace from the results
                passages = [p.strip() for p in passages]

                # Get metadata
                s3_loc=result['location']['s3Location']['uri']
                doc_name = s3_loc.replace(self.s3_doc_loc, "")
                key = "{}.metadata.json".format(doc_name)

                # Read the file from S3
                response_docread = self.s3.get_object(Bucket=self.s3_bucket,
                                                      Key="{}{}".format(self.s3_path,
                                                                        key)
                                                      )
                content = response_docread['Body'].read().decode('utf-8')

                # Parse JSON into dictionary
                metadata_dict = json.loads(content)
                source_type = metadata_dict["metadataAttributes"]["source_type"]

                res_dict = dict(query=query,
                                search_res=result['content']['text'],
                                passages=passages,
                                score=result['score'],
                                s3_loc=result['location']['s3Location']['uri'],
                                source_type=source_type,
                                doc_metadata=metadata_dict
                                )

                search_results.append(res_dict)

                time.sleep(self.search_sleep_time)

        # Put the detailed retrieval results into a dataframe
        self.df_ret_res = pd.DataFrame(search_results)

        # Aggregate results by query
        self.aggregate_retrieval_results()

    def aggregate_retrieval_results(self):
        """
        Aggregate retrieval results by query
        :param queries:
        :return:
        """

        q_rows = []
        for query in self.df_ret_res["query"].unique():

            mask = self.df_ret_res["query"] == query

            doc_value_cnts = self.df_ret_res[mask]["s3_loc"].value_counts()

            # Get passages
            pass_dict = {}
            for idx in self.df_ret_res[mask].index:
                doc_filename = self.df_ret_res.loc[idx, "s3_loc"].replace(self.s3_doc_loc,"")

                if doc_filename in pass_dict:
                    pass_dict[doc_filename] = pass_dict[doc_filename] + self.df_ret_res.loc[idx, "passages"]
                else:
                    pass_dict[doc_filename] = self.df_ret_res.loc[idx, "passages"]

            for df in pass_dict:
                pass_dict[df] = list(set(pass_dict[df]))
                pass_dict[df].sort()

            doc_dict = {}
            for s3_loc_idx in doc_value_cnts.index:
                ikey = s3_loc_idx.replace(self.s3_doc_loc, "")
                doc_dict[ikey] = int(doc_value_cnts[s3_loc_idx])

            doc_count = len(doc_dict)

            max_score = self.df_ret_res[mask]["score"].max()

            source_types = self.df_ret_res[mask]["source_type"].unique().tolist()

            q_rows.append(dict(query=query,
                               doc_count=doc_count,
                               max_score=max_score,
                               doc_scores=doc_dict,
                               passages=pass_dict,
                               source_types=source_types
                               )
                          )

        self.df_query_finds = pd.DataFrame(q_rows)

    def _build_kb_filter(self, metadata_filters: dict):
        """Helper to construct the Bedrock filter expression with type safety."""
        if not metadata_filters:
            return None

        filter_list = []
        for k, v in metadata_filters.items():
            if v is not None:
                # Ensure v is a list
                if not isinstance(v, list):
                    v = [v]

                # Ensure all elements in the list are strings
                v_string_list = [str(item) for item in v]

                filter_list.append({
                    'in': {
                        'key': k,
                        'value': v_string_list
                    }
                })

        if len(filter_list) > 1:
            return {'and': filter_list}
        return filter_list[0] if filter_list else None

    def retrieve_and_generate_results(self,
                                      queries: list,
                                      metadata_filters: dict = None):
        """
        Retrieve and Generate AI responses using dynamic metadata filtering.

        :param metadata_filters: Dict like {'source_type': ['A'], 'document_index': [123]}
        """

        # 1. Construct the filter expression
        filter_expr = self._build_kb_filter(metadata_filters)

        # 2. Build the base configuration
        kb_config = {
            'knowledgeBaseId': self.kb_id,
            'modelArn': self.model_arn
        }

        # 3. Add retrievalConfiguration only if filters or custom result counts are needed
        if filter_expr or hasattr(self, 'num_srch_res'):
            kb_config['retrievalConfiguration'] = {
                'vectorSearchConfiguration': {
                    'numberOfResults': self.num_srch_res
                }
            }
            if filter_expr:
                kb_config['retrievalConfiguration']['vectorSearchConfiguration']['filter'] = filter_expr

        ret_gen_config = {
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': kb_config
        }

        search_results = []
        for query in queries:
            rag_response = self.br_rt.retrieve_and_generate(
                input={'text': query},
                retrieveAndGenerateConfiguration=ret_gen_config
            )

            self.rag_responses.append(rag_response)

            # Collect citations using list comprehension for brevity
            citations_list = [
                ref['location']['s3Location']['uri']
                for citation in rag_response.get('citations', [])
                for ref in citation.get('retrievedReferences', [])
                if 's3Location' in ref.get('location', {})
            ]

            search_results.append({
                'query': query,
                'rag_answer': rag_response['output']['text'],
                'rag_citations': list(set(citations_list)) # Set removes duplicates from multiple snippets
            })

            time.sleep(self.ai_sleep_time)

        self.df_rag_res = pd.DataFrame(search_results)

## Read test data

In [1]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))


NameError: name 'pd' is not defined

## Build question-answer and question-source-doc maps for single passage answers

In [10]:
# Question to source document index
sp_q_doc_index_map = {k:v for k, v in zip(df_spqs["question"],
                                          df_spqs["document_index"])}
sp_q_doc_index_map = {k: "doc_{}.txt".format(v) for k, v in zip(sp_q_doc_index_map.keys(),
                                                                sp_q_doc_index_map.values())}

# Question to source document index
sp_q_answer_map = {k:v for k, v in zip(df_spqs["question"],
                                       df_spqs["answer"])}
# sp_q_doc_index_map
# sp_q_answer_map


## Get available foundational models

In [15]:
# Set up some parameters
aws_profile = "ns-admin"
aws_region = 'us-east-2'
kb_id = "DFBFV6YBJP"


In [12]:
# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
srch_analyzer.get_bedrock_foundational_models()


In [13]:
# srch_analyzer.bedrock_models.loc[[20]].T
# srch_analyzer.bedrock_models.loc[[31]].T
srch_analyzer.bedrock_models.head()

# srch_analyzer.response_models



,model_name,model_id,model_provider,input_modalities,output_modalities,infer_types,model_lifecycle
0,Nova Premier,amazon.nova-premier-v1:0:mm,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."
1,Nova Micro,amazon.nova-micro-v1:0,Amazon,[TEXT],[TEXT],[INFERENCE_PROFILE],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-1..."
2,Titan Text Embeddings V2,amazon.titan-embed-text-v2:0,Amazon,[TEXT],[EMBEDDING],[ON_DEMAND],"{'status': 'ACTIVE', 'startOfLifeTime': 2024-0..."
3,Nova Premier,amazon.nova-premier-v1:0:8k,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."
4,Nova Premier,amazon.nova-premier-v1:0:20k,Amazon,"[TEXT, IMAGE, VIDEO]",[TEXT],[],"{'status': 'LEGACY', 'endOfLifeTime': 2026-09-..."


## Retrieve Search Results

In [31]:
# Set up retriever object
srch_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Search documents for text related to queries without filtering
srch_analyzer.retrieve_query_results(queries=queries[:5])

# Search documents for text related to queries with filtering
# srch_analyzer.retrieve_query_results(queries=queries,
#                                      source_types=["gaming"])

In [32]:
# Add source document and answer columns
df_query_finds = srch_analyzer.df_query_finds.copy(deep=True)
df_query_finds["source_document_index"] = df_query_finds["query"].map(sp_q_doc_index_map)
df_query_finds["answer"] = df_query_finds["query"].map(sp_q_answer_map)


srch_analyzer.df_ret_res
srch_analyzer.df_query_finds
# srch_analyzer.response


# sp_q_doc_index_map
# sp_q_answer_map

qf_cols = ['query', 'doc_count', 'max_score', 'doc_scores',
           'source_document_index', 'answer', 'source_types', 'passages']
df_query_finds = df_query_finds[qf_cols]
df_query_finds


,query,doc_count,max_score,doc_scores,source_document_index,answer,source_types,passages
0,What do keybullet kin drop?,1,0.708868,{'doc_0.txt': 5},doc_0.txt,Keybullet kin drop a key upon death.,[gaming],"{'doc_0.txt': ['1', '2', '23', '24', '25', '26..."
1,What kind of gun does the bandana bullet kin use?,1,0.579380,{'doc_0.txt': 5},doc_0.txt,The bandana bullet kin wields a machine pistol.,[gaming],"{'doc_0.txt': ['1', '10', '11', '12', '13', '1..."
2,What do the giants look like?,3,0.508541,"{'doc_1.txt': 2, 'doc_15.txt': 2, 'doc_0.txt': 1}",doc_1.txt,"One giant is burly, grey-skinned, and 20 feet ...","[gaming, entertainment]","{'doc_1.txt': ['10', '11', '12', '13', '14', '..."
3,What happens on day 2?,3,0.376101,"{'doc_18.txt': 2, 'doc_16.txt': 2, 'doc_9.txt'...",doc_1.txt,"After a few miles of winding tunnel, you emerg...","[entertainment, gaming]","{'doc_18.txt': ['77', '78'], 'doc_16.txt': [],..."
4,What were the requirements for the project?,5,0.388976,"{'doc_5.txt': 1, 'doc_2.txt': 1, 'doc_6.txt': ...",doc_2.txt,The tool had the following requirements:\n- Ch...,"[data_science, government, entertainment]","{'doc_5.txt': ['38', '39'], 'doc_2.txt': ['13'..."


## Retrieve and generate AI responses using RAG

In [11]:
# Set up retriever object
rag_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                   aws_region=aws_region,
                                   kb_id=kb_id)

# Set queries
queries = df_spqs["question"].tolist()

# Retrieve RAG responses for text related to queries without filtering
rag_analyzer.retrieve_and_generate_results(queries=queries)



In [15]:
df_rag_res_ans = rag_analyzer.df_rag_res.copy(deep=True)
df_rag_res_ans["source_document_index"] = df_rag_res_ans["query"].map(sp_q_doc_index_map)
df_rag_res_ans["answer"] = df_rag_res_ans["query"].map(sp_q_answer_map)


dra_cols = ['query', 'rag_answer', 'answer', 'rag_citations', 'source_document_index']
df_rag_res_ans = df_rag_res_ans[dra_cols]

df_rag_res_ans


,query,rag_answer,answer,rag_citations,source_document_index
0,What do keybullet kin drop?,Keybullet Kin drop a key upon death. Jammed Ke...,Keybullet kin drop a key upon death.,[s3://rag-search-tests/documents/doc_0.txt],doc_0.txt
1,What kind of gun does the bandana bullet kin use?,The Bandana Bullet Kin uses a Machine Pistol.,The bandana bullet kin wields a machine pistol.,[s3://rag-search-tests/documents/doc_0.txt],doc_0.txt
2,What do the giants look like?,There are two giants described. The first gian...,"One giant is burly, grey-skinned, and 20 feet ...","[s3://rag-search-tests/documents/doc_1.txt, s3...",doc_1.txt
3,What happens on day 2?,"Sorry, I am unable to assist you with this req...","After a few miles of winding tunnel, you emerg...",[],doc_1.txt
4,What were the requirements for the project?,The required features for the project were:\n-...,The tool had the following requirements:\n- Ch...,[s3://rag-search-tests/documents/doc_2.txt],doc_2.txt
5,What data did was used to test the prototype?,The prototype was not tested with any specific...,Grace Hopper's Wikipedia page and Alan Turing'...,"[s3://rag-search-tests/documents/doc_2.txt, s3...",doc_2.txt
6,How do the data storage options compare?,The data storage options are organized into di...,For fast start: use SQLite3 and ChromaDB (File...,[s3://rag-search-tests/documents/doc_3.txt],doc_3.txt
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,UTF-8 support was added for European languages...,[s3://rag-search-tests/documents/doc_3.txt],doc_3.txt
8,How do I make a button?,"To make a button in marimo, use `mo.ui.button(...",import marimo as mo\n\nbutton = mo.ui.run_butt...,"[s3://rag-search-tests/documents/doc_4.txt, s3...",doc_4.txt
9,When might I use caching?,You might use caching when you have expensive ...,"You might use caching when, for example, your ...",[s3://rag-search-tests/documents/doc_4.txt],doc_4.txt


## Filter, retrieve and generate AI responses using RAG

In [6]:
df_spqs


,document_index,question,answer
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ..."
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg..."
4,2,What were the requirements for the project?,The tool had the following requirements:\n- Ch...
5,2,What data did was used to test the prototype?,Grace Hopper's Wikipedia page and Alan Turing'...
6,3,How do the data storage options compare?,For fast start: use SQLite3 and ChromaDB (File...
7,3,When was UTF-8 support added for European lang...,UTF-8 support was added for European languages...
8,4,How do I make a button?,import marimo as mo\n\nbutton = mo.ui.run_butt...
9,4,When might I use caching?,"You might use caching when, for example, your ..."


In [37]:
q_idxs = [3, 10, 23, 27, 39]

filtered_results = []
for idx in q_idxs:

    # Get the query
    queries = [df_spqs.loc[idx, "question"]]

    # Create a metadata filter
    metadata_filters = {"document_index": "doc_{}".format(df_spqs.loc[idx, "document_index"])}

    # Set up retriever object
    rag_analyzer = BedrockKBRetriever(aws_profile=aws_profile,
                                       aws_region=aws_region,
                                       kb_id=kb_id)

    # Search RAG responses for text related to queries with filtering
    rag_analyzer.retrieve_and_generate_results(queries=queries,
                                               metadata_filters=metadata_filters)

    # Enrich the results dataframe
    df_rag_res_ans = rag_analyzer.df_rag_res.copy(deep=True)
    df_rag_res_ans["source_document_index"] = df_rag_res_ans["query"].map(sp_q_doc_index_map)
    df_rag_res_ans["answer"] = df_rag_res_ans["query"].map(sp_q_answer_map)

    # rearrange columns
    dra_cols = ['query', 'rag_answer', 'answer', 'rag_citations', 'source_document_index']
    df_rag_res_ans = df_rag_res_ans[dra_cols]

    # Add this to a list of dataframes
    filtered_results.append(df_rag_res_ans)



{'document_index': 'doc_1'}
{'document_index': 'doc_5'}
{'document_index': 'doc_11'}
{'document_index': 'doc_13'}
{'document_index': 'doc_19'}


In [39]:
filtered_results[1]

df_rag_filt = pd.concat(objs=filtered_results)
df_rag_filt.index = q_idxs
df_rag_filt


,query,rag_answer,answer,rag_citations,source_document_index
3,What happens on day 2?,"On day 2, after traveling a few miles through ...","After a few miles of winding tunnel, you emerg...",[s3://rag-search-tests/documents/doc_1.txt],doc_1.txt
10,What are the key topics of this article?,The article focuses on how Data Scientists can...,"The key topics of this article are: ""why prior...",[s3://rag-search-tests/documents/doc_5.txt],doc_5.txt
23,For what work did I receive criticism for my r...,You received criticism for your work on sparse...,You received criticism for your research on sp...,[s3://rag-search-tests/documents/doc_11.txt],doc_11.txt
27,How can I freeze a variable during training in...,"Based on the search results provided, there is...","To freeze an attribute during training, you ca...",[s3://rag-search-tests/documents/doc_13.txt],doc_13.txt
39,In what way can including unrelated documents ...,Including unrelated documents can negatively i...,"Noise Power (Cuconasu et al., 2024) provide a ...",[s3://rag-search-tests/documents/doc_19.txt],doc_19.txt
